In [76]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

from secretstuff.secret import *
from pymongo import MongoClient
import random
from bson import ObjectId
import openai
from openai import OpenAI
import torch
from services.mongodb import catalogue
import json
import ast
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from typing import Literal
from services.mongodb import catalogue, CatalogueItem
from pydantic import BaseModel
import numpy as np


In [77]:
client = MongoClient(MONGODB_CONNECTION_STRING)
db = client.kagame
wardrobe = db.wardrobe
users = db.users

In [3]:
openai_client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJ_ID)

In [23]:
# style info
styles = ['classic and timeless', 'elegant', 'casual', 'streetwear', 'minimalist', 'bohemian', 'vintage', 'retro', 'preppy', 'athleisure',
          'glamourous', 'business or business casual', 'artsy or avant-garde', 'edgy and emo', 'rock and punk', 'grunge', 'cottagecore', 'y2k', 
          'romantic', 'military/utility', 'western/cowboy', 'goth', 'androgynous', 'beachwear', 'academia']
style = random.choice(styles)
style

'glamourous'

In [18]:
# choose a random item
random_item = wardrobe.aggregate([{"$sample": {"size": 1}}])
random_item_chosen = list(random_item)[0] if random_item else None
name = random_item_chosen['name']
category = random_item_chosen['category']
tags = random_item_chosen['tags']
print(name, category, tags)

Blue Gingham Button-Up Tops ['shirt', 'fitted', 'short sleeve', 'casual', 'daytime']


In [35]:
# additional query embeddings
def generate_query_embedding(query, client):
    response = client.embeddings.create(input=query, model="text-embedding-3-large").data[0].embedding
    return np.array(response)

query = "Looking for a fancy outfit"
query_embedding = generate_query_embedding(query, openai_client)


In [54]:
# combined query vector
q_v = 0.16 * np.array(random_item_chosen['tags_embed'][0]) + 0.16 * np.array(random_item_chosen['tags_embed'][1]) + 0.16 * np.array(random_item_chosen['tags_embed'][2]) + 0.16 * np.array(random_item_chosen['tags_embed'][3]) + 0.16 * np.array(random_item_chosen['tags_embed'][4]) + 0.2 * query_embedding
q_v = np.array(q_v, dtype=np.float64).tolist()

In [57]:
len(q_v)

3072

In [87]:
limit = 100

pipeline = [
    {
        '$vectorSearch': {
            'index': 'vector_index',
            'path': 'tags_embed.0',
            'queryVector': q_v,
            'numCandidates': limit,
            'limit': limit,
        }
    },
    {
        "$addFields": {
            "debug_check": {"$arrayElemAt": ["$tags_embed", 1]},
            "other_score1": {
                "$sum": {
                    "$map": {
                        "input": {"$range": [0, {"$size": {"$arrayElemAt": ["$tags_embed", 1]}}]},
                        "as": "index",
                        "in": {
                            "$multiply": [
                                {"$arrayElemAt": ["tags_embed.1", "$$index"]},
                                {"$arrayElemAt": [q_v, "$$index"]}
                            ]
                        }
                    }
                }
            }
        }
    },
    {
        "$addFields": {
            "other_score2": {
                "$sum": {
                    "$map": {
                        "input": {"$range": [0, {"$arrayElemAt": ["$tags_embed", 2]}]},
                        "as": "index",
                        "in": {
                            "$multiply": [
                                {"$arrayElemAt": ["tags_embed.2", "$$index"]},
                                {"$arrayElemAt": [q_v, "$$index"]}
                            ]
                        }
                    }
                }
            }
        }
    },
    {
        "$addFields": {
            "other_score3": {
                "$sum": {
                    "$map": {
                        "input": {"$range": [0, {"$arrayElemAt": ["$tags_embed", 3]}]},
                        "as": "index",
                        "in": {
                            "$multiply": [
                                {"$arrayElemAt": ["tags_embed.3", "$$index"]},
                                {"$arrayElemAt": [q_v, "$$index"]}
                            ]
                        }
                    }
                }
            }
        }
    },
    {
        "$addFields": {
            "other_score4": {
                "$sum": {
                    "$map": {
                        "input": {"$range": [0, {"$arrayElemAt": ["$tags_embed", 4]}]},
                        "as": "index",
                        "in": {
                            "$multiply": [
                                {"$arrayElemAt": ["tags_embed.4", "$$index"]},
                                {"$arrayElemAt": [q_v, "$$index"]}
                            ]
                        }
                    }
                }
            }
        }
    },
    {
        "$addFields": {
            "combined_score": {
                "$add": [
                    {"$multiply": [0.2, {"$meta": "vectorSearchScore"}]},
                    {"$multiply": [0.2, "$other_score1"]},
                    {"$multiply": [0.2, "$other_score2"]},
                    {"$multiply": [0.2, "$other_score3"]},
                    {"$multiply": [0.2, "$other_score4"]}
                ]
            }
        }
    },
    {
        "$sort": { "combined_score": -1 }
    },
    {
        "$limit": 5
    },
    {
        '$project': {
            'name': 1,
            'category': 1,
            'tags': 1,
            "type_score": { "$meta": "vectorSearchScore" }
        }
    }
]

result = wardrobe.aggregate(pipeline)
for i in result:
    print(i)

In [86]:
result.to_list()

[]

In [ ]:
# generation